# Day 18 — JSON Parsing & **OutputFixingParser**

**Phase 1 · Module 2: AWS Bedrock & AgentCore · Capstone** · ~30 min

This morning feeds the afternoon capstone. A RAG agent's answer is only useful if your code can *read* it — and models emit JSON that is (a) **streamed** token-by-token, and (b) sometimes **malformed** (a trailing comma, a missing brace, prose wrapped around it). Three tools cover the whole gap:

- **`ijson`** — parse a JSON *stream* incrementally, before the whole payload has arrived.
- **`JsonOutputParser`** (LangChain) — turn a model's text into a validated Python object, optionally against a Pydantic schema.
- **`OutputFixingParser`** — when parsing fails, send the broken output *back to a model* to repair, then re-parse.

> **Runnable keyless:** no API key. The "fixer" model is a deterministic `FakeListChatModel` that returns corrected JSON — same interface a real Bedrock Claude would expose.


## 1. The problem — JSON that isn't ready, or isn't valid

A loan agent returns a structured verdict. Two things go wrong in production:

1. The transport **streams** it — you receive `{"decis` … `ion": "appr` … one chunk at a time.
2. The model **fumbles** the syntax — a stray trailing comma, a code fence, an apology before the object.

`json.loads` needs the *complete, valid* string. Both cases break it.

In [1]:
import json

streamed = '{"decision": "approved", "risk": "low"}'
# imagine this arriving in pieces; json.loads can't start until it's whole
print(json.loads(streamed))

broken = '```json\n{"decision": "approved", "risk": "low",}\n```'   # fence + trailing comma
try:
    json.loads(broken)
except json.JSONDecodeError as e:
    print("json.loads failed:", e)

{'decision': 'approved', 'risk': 'low'}
json.loads failed: Expecting value: line 1 column 1 (char 0)


☝ `json.loads` is all-or-nothing: it needs the entire payload and rejects the smallest syntax slip. Real model output violates both assumptions, so we need incremental parsing *and* repair.

## 2. `ijson` — parse the stream as it arrives

`ijson` is an **incremental** (SAX-style) JSON parser. Instead of loading everything into memory then parsing, it emits **events** as bytes flow in — `start_map`, `map_key`, `string`, `number`, `end_map`. Useful when the LLM is streaming a large structured answer and you want to react to fields *before* the object is finished.

In [2]:
import ijson, io

# a big-ish streamed policy response arriving as bytes
payload = b'''{"query": "max LTV for buy-to-let?",
  "answer": "75% loan-to-value for buy-to-let mortgages.",
  "citations": ["POL-014", "POL-021"],
  "confidence": 0.92}'''

# ijson.parse yields (prefix, event, value) as the bytes are consumed
for prefix, event, value in ijson.parse(io.BytesIO(payload)):
    print(f"{prefix:12} {event:10} {value}")

             start_map  None
             map_key    query
query        string     max LTV for buy-to-let?
             map_key    answer
answer       string     75% loan-to-value for buy-to-let mortgages.
             map_key    citations
citations    start_array None
citations.item string     POL-014
citations.item string     POL-021
citations    end_array  None
             map_key    confidence
confidence   number     0.92
             end_map    None


☝ Each field surfaces as an **event** the instant it's parsed — no waiting for `end_map`. `prefix` names the path (`citations.item` for array elements), `event` is the token type, `value` the payload.

In [3]:
# Higher-level: pull just the citations array, item by item, from the stream
citations = list(ijson.items(io.BytesIO(payload), "citations.item"))
print("citations:", citations)

# and grab a single scalar without building the whole dict
conf = next(ijson.items(io.BytesIO(payload), "confidence"))
print("confidence:", conf)

citations: ['POL-014', 'POL-021']
confidence: 0.92


☝ `ijson.items(stream, path)` yields fully-formed Python objects at a JSON path — `citations.item` gives each array element. This is how you'd show sources in a UI the moment they stream in, before the answer finishes.

## 3. `JsonOutputParser` — model text → dict

LangChain's `JsonOutputParser` takes raw model output and returns a Python dict. Bind it to a **Pydantic** schema and it produces *format instructions* to put in the prompt. Note: it uses the schema to build those instructions, but does **not** validate the output against it (see §6) — parsing valid-but-wrong-shape JSON succeeds silently.

In [4]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class LoanVerdict(BaseModel):
    decision: str      = Field(description="approved | declined | referred")
    risk:     str      = Field(description="low | medium | high")
    reason:   str      = Field(description="one-sentence rationale")

parser = JsonOutputParser(pydantic_object=LoanVerdict)

# the instructions you'd inject into the prompt so the model emits the right shape
print(parser.get_format_instructions()[:200], "...\n")

model_text = '{"decision": "referred", "risk": "medium", "reason": "income unverified"}'
print(parser.parse(model_text))

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fe ...

{'decision': 'referred', 'risk': 'medium', 'reason': 'income unverified'}


☝ `JsonOutputParser` also tolerates a **code fence** around the JSON (it strips ```` ```json ````), so it's more forgiving than raw `json.loads`. But it still can't fix genuinely broken syntax — that's the next tool. It even parses *partial* JSON when streaming, so a UI can update mid-generation.

## 4. `OutputFixingParser` — auto-repair the unparseable

When output is broken beyond the base parser's tolerance, `OutputFixingParser` wraps it: on a parse failure it sends the bad text **plus the error** to a model with a "fix this" prompt, then re-parses the corrected result. It needs an LLM — here a deterministic `FakeListChatModel` standing in for Bedrock Claude, so the notebook runs keyless.

In [5]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_classic.output_parsers import OutputFixingParser

# the "repair" model: returns clean JSON regardless of the broken input (deterministic stand-in)
fixer_llm = FakeListChatModel(responses=[
    '{"decision": "approved", "risk": "low", "reason": "strong income and clean history"}'
])

fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=fixer_llm)

garbled = "Sure! Here's the verdict:\n{'decision': 'approved', 'risk': 'low' 'reason': strong income}"
print("repaired ->", fixing_parser.parse(garbled))

repaired -> {'decision': 'approved', 'risk': 'low', 'reason': 'strong income and clean history'}


☝ The base `JsonOutputParser` chokes on that (single quotes, missing comma, unquoted value), so `OutputFixingParser` routes it to the fixer model and re-parses the clean result. Swap `fixer_llm` for a real `ChatBedrock` and the flow is identical — one line changes.

## 5. Putting it together — a robust extraction step

The capstone agent's post-processing: try the fast path, fall back to repair only when needed. Cheap when the model behaves, resilient when it doesn't.

In [6]:
def extract_verdict(raw: str) -> dict:
    try:
        return parser.parse(raw)              # fast path — clean JSON
    except Exception:
        return fixing_parser.parse(raw)       # repair path — costs one extra model call

print(extract_verdict('{"decision": "declined", "risk": "high", "reason": "DTI over limit"}'))
print(extract_verdict("junk {'decision': 'approved'  'risk':'low' 'reason': ok}"))

{'decision': 'declined', 'risk': 'high', 'reason': 'DTI over limit'}
{'decision': 'approved', 'risk': 'low', 'reason': 'strong income and clean history'}


☝ Guard the expensive repair behind a `try` — most calls take the free fast path. This is exactly the pattern the Day-18 capstone uses to turn a streamed Bedrock answer into a typed record your app can trust.

## 6. When it fails — the limits of each tool

None of these are magic. Three failure modes to guard in production: the **fixer** can itself return junk, `JsonOutputParser` does **not** enforce the schema (only shapes the prompt), and `ijson` **raises** on a truncated stream rather than returning a partial.

In [7]:
import ijson, io

# FAIL 1 — the FIXER model itself returns junk. OutputFixingParser re-parses once, still raises.
bad_fixer = FakeListChatModel(responses=["I think it was approved, not sure about risk"])
brittle = OutputFixingParser.from_llm(parser=parser, llm=bad_fixer, max_retries=1)
try:
    brittle.parse("{'decision': 'approved' 'risk' low}")
except Exception as e:
    print("FAIL 1 (bad fixer):", type(e).__name__, "-", str(e).splitlines()[0][:60])

# FAIL 2 — JsonOutputParser parses valid JSON but does NOT enforce the schema (silent).
wrong_shape = '{"decision": "approved", "danger": "low"}'      # 'reason' missing, 'danger' bogus
out = parser.parse(wrong_shape)
print("FAIL 2 (no schema check):", out)
try:
    out["reason"]                                              # the KeyError, one step too late
except KeyError as e:
    print("   -> downstream KeyError:", e)
print("   real validation ->", end=" ")
try:
    LoanVerdict(**out)                                         # THIS enforces the shape
except Exception as e:
    print(type(e).__name__, "(missing 'reason')")

# FAIL 3 — ijson on a TRUNCATED stream raises; it doesn't return the partial you saw.
truncated = b'{"answer": "75% LTV", "citations": ["POL-01'
try:
    list(ijson.items(io.BytesIO(truncated), "citations.item"))
except Exception as e:
    print("FAIL 3 (truncated stream):", type(e).__name__, "-", str(e)[:45])

FAIL 1 (bad fixer): OutputParserException - Invalid json output: I think it was approved, not sure about
FAIL 2 (no schema check): {'decision': 'approved', 'danger': 'low'}
   -> downstream KeyError: 'reason'
   real validation -> ValidationError (missing 'reason')
FAIL 3 (truncated stream): IncompleteJSONError - parse error: premature EOF
                  


☝ Three ways the pipeline still bites you: **(1)** `OutputFixingParser` re-parses only `max_retries` times — a fixer that also emits junk still raises. **(2)** `JsonOutputParser` parses *valid* JSON but does **not** enforce the Pydantic schema — a wrong shape passes silently and blows up *downstream* as a `KeyError`. To actually validate, feed the dict into `LoanVerdict(**out)`. **(3)** `ijson` is incremental, not resilient — a **truncated** stream raises `IncompleteJSONError`, it doesn't hand back the partial you already saw. Guard for all three in production.

## 7. Recap — reading model output safely

| Tool | Why used |
|---|---|
| `json.loads` | baseline — needs the **whole, valid** string; breaks on streams and syntax slips |
| `ijson.parse` / `.items` | **incremental** parse — react to fields as they stream, path-addressable |
| `JsonOutputParser` | model text → dict, tolerates code fences, streams partials (does **not** validate schema) |
| Pydantic `BaseModel` | generates prompt **format instructions**; call `Model(**out)` to actually validate |
| `OutputFixingParser` | on failure, **repair** via a model call, then re-parse (bounded by `max_retries`) |
| `try` fast-path → fix | cheap when valid, resilient when not |